# Dataset Collection Control Panel

This notebook is set up for the common path without manual edits.

Use it like this:
1. Run the setup cells from top to bottom once.
2. Run the `start_default_collection()` cell.
3. Run the `watch_progress(RUN_DIR)` cell.

If you reopen Jupyter later, use the `latest_run_dir()` cell to reattach to the latest run and then watch it again.


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Markdown, clear_output, display


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'scripts' / 'collect_osu_ranked_dataset.py').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from the current notebook environment.')


REPO_ROOT = find_repo_root(Path.cwd())
RAW_DIR = REPO_ROOT / 'data' / 'raw'
COLLECTOR = REPO_ROOT / 'scripts' / 'collect_osu_ranked_dataset.py'

display(Markdown(f'**Repo root:** `{REPO_ROOT}`'))
display(Markdown(f'**Collector:** `{COLLECTOR}`'))


In [ ]:
DEFAULT_CONFIG = {
    'top_country_count': 100,
    'players_per_country': 100,
    'country_ranking_max': 10_000,
    'country_sample_mean_ratio': 0.5,
    'country_sample_std_ratio': 0.2,
    'recent_scores_per_user': 30,
    'best_scores_per_user': 20,
    'request_delay_seconds': 0.05,
    'random_seed': 42,
    'ranking_type': 'performance',
}

display(Markdown(
    '\n'.join([
        '**Default collection config**',
        '',
        *(f'- `{key}`: `{value}`' for key, value in DEFAULT_CONFIG.items()),
    ])
))


In [ ]:
def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')


def default_run_name() -> str:
    return f"osu_country_try_data_full_{utc_stamp()}"


def build_default_command(run_dir: Path) -> list[str]:
    return [
        'python',
        str(COLLECTOR),
        '--output-dir', str(run_dir),
        '--ranking-type', str(DEFAULT_CONFIG['ranking_type']),
        '--top-country-count', str(DEFAULT_CONFIG['top_country_count']),
        '--players-per-country', str(DEFAULT_CONFIG['players_per_country']),
        '--country-ranking-max', str(DEFAULT_CONFIG['country_ranking_max']),
        '--country-sample-mean-ratio', str(DEFAULT_CONFIG['country_sample_mean_ratio']),
        '--country-sample-std-ratio', str(DEFAULT_CONFIG['country_sample_std_ratio']),
        '--recent-scores-per-user', str(DEFAULT_CONFIG['recent_scores_per_user']),
        '--best-scores-per-user', str(DEFAULT_CONFIG['best_scores_per_user']),
        '--random-seed', str(DEFAULT_CONFIG['random_seed']),
        '--request-delay-seconds', str(DEFAULT_CONFIG['request_delay_seconds']),
    ]


def read_json(path: Path, default):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def tail_lines(path: Path, limit: int = 12) -> list[str]:
    if not path.exists():
        return []
    lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    return lines[-limit:]


def latest_run_dir() -> Path:
    candidates = sorted(
        [path for path in RAW_DIR.glob('osu_country_try_data_*') if path.is_dir()],
        key=lambda path: path.name,
    )
    if not candidates:
        raise RuntimeError('No existing country-seeded run directories were found under data/raw/.')
    return candidates[-1]


def start_default_collection() -> Path:
    run_dir = RAW_DIR / default_run_name()
    run_dir.mkdir(parents=True, exist_ok=True)
    stdout_path = run_dir / 'collector.stdout.log'
    stderr_path = run_dir / 'collector.stderr.log'
    command = build_default_command(run_dir)

    stdout_handle = stdout_path.open('a', encoding='utf-8')
    stderr_handle = stderr_path.open('a', encoding='utf-8')
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        stdout=stdout_handle,
        stderr=stderr_handle,
        creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0),
    )
    print('Started collector PID:', process.pid)
    print('Run dir:', run_dir)
    print('Command:')
    print(' '.join(shlex.quote(part) for part in command))
    return run_dir


def render_progress(run_dir: Path, log_tail: int = 12) -> str:
    state = read_json(run_dir / 'state.json', {})
    metadata = read_json(run_dir / 'export_metadata.json', {})
    config = read_json(run_dir / 'config.json', {})
    stdout_tail = tail_lines(run_dir / 'collector.stdout.log', limit=log_tail)
    stderr_tail = tail_lines(run_dir / 'collector.stderr.log', limit=log_tail)

    phase = state.get('phase', 'unknown')
    processed = state.get('processed_user_count', 0)
    total = state.get('total_sampled_user_count') or state.get('sampled_user_count') or 0
    progress = (processed / total * 100.0) if total else 0.0

    lines = [
        '# Collection Progress',
        '',
        f'- Run dir: `{run_dir}`',
        f'- Phase: `{phase}`',
        f'- Last updated: `{state.get("last_updated_at", "")}`',
        f'- Ranking type: `{state.get("ranking_type") or config.get("ranking_type", "")}`',
        f'- Country leaderboard size: `{state.get("top_country_total_available", "")}`',
        f'- Selected countries: `{state.get("selected_country_count", "")}`',
        f'- Users processed: `{processed}` / `{total}` ({progress:.2f}%)',
        f'- Unique sampled users: `{state.get("unique_sampled_user_count", "")}`',
        f'- Ranking pages fetched: `{state.get("ranking_pages_fetched", "")}`',
        f'- Beatmaps cached: `{state.get("cached_beatmap_count", "")}`',
        f'- Beatmaps referenced: `{state.get("referenced_beatmap_count", "")}`',
        f'- CSV rows: `{state.get("csv_row_count", metadata.get("exported_row_count", ""))}`',
        f'- Current country: `{state.get("current_country_code", "")}`',
        f'- Current page: `{state.get("current_page", "")}`',
        f'- Note: `{state.get("note", "")}`',
        '',
        '## Sampled Users Per Country',
        '',
    ]

    sampled_country_counts = state.get('sampled_country_counts') or {}
    processed_country_counts = state.get('processed_country_counts') or {}
    if sampled_country_counts:
        lines.extend(f'- `{country}`: `{count}`' for country, count in sorted(sampled_country_counts.items()))
    else:
        lines.append('- no data yet')

    lines.extend(['', '## Processed Users Per Country', ''])
    if processed_country_counts:
        lines.extend(f'- `{country}`: `{count}`' for country, count in sorted(processed_country_counts.items()))
    else:
        lines.append('- no data yet')

    lines.extend(['', '## Collector Stdout Tail', ''])
    if stdout_tail:
        lines.extend(f'- `{line}`' for line in stdout_tail)
    else:
        lines.append('- no stdout log yet')

    lines.extend(['', '## Collector Stderr Tail', ''])
    if stderr_tail:
        lines.extend(f'- `{line}`' for line in stderr_tail)
    else:
        lines.append('- no stderr log yet')

    return '\n'.join(lines)


def show_progress(run_dir: Path) -> None:
    display(Markdown(render_progress(run_dir)))


def watch_progress(run_dir: Path, refresh_seconds: float = 5.0, max_refreshes: int | None = None) -> None:
    iteration = 0
    while True:
        clear_output(wait=True)
        show_progress(run_dir)
        state = read_json(run_dir / 'state.json', {})
        if state.get('phase') == 'done':
            break
        iteration += 1
        if max_refreshes is not None and iteration >= max_refreshes:
            break
        time.sleep(refresh_seconds)


## Start New Default Run

Run the next cell as-is. It creates a fresh run directory with the default config and starts the collector in the background.


In [ ]:
RUN_DIR = start_default_collection()
RUN_DIR


## Watch Current Run

Run the next cell after the start cell. It refreshes the progress panel until the collector finishes.


In [ ]:
watch_progress(RUN_DIR, refresh_seconds=5)


## Reattach To Latest Existing Run

If you restarted the notebook kernel or reopened Jupyter, run the next two cells to attach to the latest run and watch it again.


In [ ]:
RUN_DIR = latest_run_dir()
show_progress(RUN_DIR)
RUN_DIR


In [ ]:
watch_progress(RUN_DIR, refresh_seconds=5)
